# AI Agents

In [2]:
import getpass
import os
env_path = r"C:\Users\vaalt\OneDrive\Desktop\Projects\Eventi speaker\Packt Bootcamp\code\.env"
from dotenv import load_dotenv
load_dotenv(dotenv_path=env_path, override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [8]:
import os
from langchain_core.messages import HumanMessage
import requests
#from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
#from langchain.tools import BaseTool, StructuredTool, tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

from pathlib import Path


from langchain_openai import ChatOpenAI
# api_key = os.environ.get("OPENAI_API_KEY")
llm = ChatOpenAI(
    model="gpt-5.2",
    #api_key=openai_api_key,
)


llm.invoke("tell me a joke")


AIMessage(content='Why don’t skeletons fight each other?\n\nThey don’t have the guts.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 10, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.2-2025-12-11', 'system_fingerprint': None, 'id': 'chatcmpl-DIYK1zQPF523s5So9IhgrgfJcZzIx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ce1c3-6ff2-71e2-84d1-1cd792823779-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 19, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [4]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    #api_key=openai_api_key
    # With the `text-embedding-3` class
    # of models, you can specify the size
    # of the embeddings you want returned.
    # dimensions=1024
)

## SQL Agent

In [3]:
import pandas as pd
import sqlite3
import random

# ----------------------------
# 1. Product data
# ----------------------------
data = {
    "Product ID": [f"P{str(i).zfill(3)}" for i in range(1, 31)],
    "Product Name": [
        "Classic Piadina", "Vegetarian Piadina", "Ham & Cheese Piadina", 
        "Nutella Piadina", "Caprese Salad", "Tigelle Bread", 
        "Sun-Dried Tomatoes", "Parmesan Cheese", "Prosciutto Crudo", 
        "Balsamic Vinegar", "Truffle Piadina", "Spicy Salami Piadina", 
        "Vegan Piadina", "Chicken Caesar Salad", "Garlic Bread", 
        "Olive Tapenade", "Ricotta Cheese", "Mortadella", 
        "Pesto Sauce", "Marinated Artichokes", "Four Cheese Piadina", 
        "Grilled Eggplant", "Smoked Salmon Piadina", "Tuna Salad", 
        "Rosemary Focaccia", "Anchovy Spread", "Gorgonzola Cheese", 
        "Speck", "Fig Jam", "Lemon Olive Oil"
    ],
    "Category": [
        "Piadina", "Piadina", "Piadina", "Dessert", "Salad", "Bread", 
        "Condiment", "Cheese", "Meat", "Condiment", "Piadina", "Piadina", 
        "Piadina", "Salad", "Bread", "Condiment", "Cheese", "Meat", 
        "Condiment", "Condiment", "Piadina", "Vegetable", "Piadina", 
        "Salad", "Bread", "Condiment", "Cheese", "Meat", "Condiment", 
        "Condiment"
    ],
    "Price": [
        5.50, 6.00, 6.50, 4.00, 7.00, 3.50, 4.50, 8.00, 9.00, 6.00, 
        7.50, 6.75, 6.25, 8.50, 3.75, 5.25, 7.75, 8.50, 4.75, 5.50, 
        7.00, 4.25, 9.50, 7.25, 4.00, 5.75, 8.25, 9.25, 6.50, 5.00
    ],
    "Stock": [
        20, 15, 25, 10, 12, 30, 40, 18, 22, 25, 14, 16, 20, 10, 35, 
        28, 15, 12, 30, 25, 18, 22, 14, 10, 40, 20, 15, 12, 18, 25
    ],
    "Allergens": [
        "Gluten, Dairy", "Gluten, Dairy", "Gluten, Dairy, Pork", 
        "Gluten, Dairy, Nuts", "Dairy", "Gluten", "None", "Dairy", 
        "Pork", "None", "Gluten, Dairy, Truffle", "Gluten, Dairy, Pork", 
        "Gluten", "Dairy, Eggs", "Gluten", "None", "Dairy", "Pork", 
        "Nuts", "None", "Gluten, Dairy", "None", "Gluten, Dairy, Fish", 
        "Fish, Eggs", "Gluten", "Fish", "Dairy", "Pork", "None", "None"
    ],
    "Description": [
        "Traditional Italian flatbread filled with prosciutto and cheese.",
        "Flatbread filled with grilled vegetables and mozzarella.",
        "Flatbread filled with ham and melted cheese.",
        "Sweet flatbread filled with Nutella and crushed hazelnuts.",
        "Fresh salad with mozzarella, tomatoes, and basil.",
        "Soft round bread, perfect for pairing with spreads or cured meats.",
        "Rich and tangy sun-dried tomatoes, ideal for salads or toppings.",
        "Aged Parmesan cheese, perfect for grating over dishes.",
        "Thinly sliced cured ham, a classic Italian delicacy.",
        "Authentic balsamic vinegar from Modena, great for salads and marinades.",
        "Flatbread filled with truffle cream and cheese.",
        "Flatbread filled with spicy salami and mozzarella.",
        "Flatbread filled with vegan cheese and grilled vegetables.",
        "Crisp romaine lettuce with grilled chicken and Caesar dressing.",
        "Toasted bread with garlic and olive oil.",
        "Savory olive spread, perfect for appetizers.",
        "Creamy ricotta cheese, ideal for desserts or pasta.",
        "Thinly sliced mortadella, a flavorful Italian sausage.",
        "Fresh basil pesto sauce, great for pasta or sandwiches.",
        "Marinated artichokes, perfect for antipasti platters.",
        "Flatbread filled with a blend of four Italian cheeses.",
        "Grilled eggplant slices, seasoned with olive oil and herbs.",
        "Flatbread filled with smoked salmon and cream cheese.",
        "Fresh tuna salad with celery and mayonnaise.",
        "Soft focaccia bread infused with rosemary.",
        "Savory anchovy spread, ideal for bruschetta.",
        "Creamy Gorgonzola cheese, perfect for sauces or salads.",
        "Smoky cured speck, a northern Italian specialty.",
        "Sweet fig jam, great for pairing with cheese.",
        "Lemon-infused olive oil, perfect for dressings or marinades."
    ],
    "Calories": [
        350, 300, 400, 450, 250, 200, 50, 110, 150, 20, 380, 420, 310, 
        270, 180, 90, 120, 160, 80, 70, 400, 60, 450, 320, 220, 40, 
        130, 170, 100, 30
    ],
    "Vegetarian": [
        False, True, False, False, True, True, True, True, False, True, 
        True, False, True, False, True, True, True, False, True, True, 
        True, True, False, False, True, False, True, False, True, True
    ]
}

# ----------------------------
# 2. Supplier data
# ----------------------------
suppliers = {
    "Supplier ID": [f"S{str(i).zfill(3)}" for i in range(1, 11)],
    "Supplier Name": [
        "PiadaMaster Inc.", "Verde Farms", "Dolce Italia", 
        "Formaggi & Co.", "Salumi Rossi", "Mediterraneo Ltd.", 
        "Truffle Treasures", "Balsamico Tradizionale", 
        "Pane Fresco", "Olio di Sole"
    ],
    "Contact Email": [
        "contact@piadamaster.com", "info@verdefarms.com", 
        "orders@dolceitalia.it", "sales@formaggico.com", 
        "hello@salumirossi.it", "support@mediterraneo.com", 
        "info@truffletreasures.com", "shop@balsamicotradizionale.it", 
        "contact@panefresco.com", "service@oliodisole.it"
    ],
    "Country": [
        "Italy", "Italy", "Italy", "Italy", "Italy", 
        "Greece", "Italy", "Italy", "Italy", "Spain"
    ]
}

# ----------------------------
# 3. Create DataFrames
# ----------------------------
products_df = pd.DataFrame(data)
suppliers_df = pd.DataFrame(suppliers)

# ----------------------------
# 4. Assign a random Supplier ID to each product
# ----------------------------
supplier_ids = suppliers_df["Supplier ID"].tolist()
products_df["Supplier ID"] = [random.choice(supplier_ids) for _ in range(len(products_df))]

# ----------------------------
# 5. Save to SQLite
# ----------------------------
conn = sqlite3.connect("piadineria.db")

products_df.to_sql("products", conn, if_exists="replace", index=False)
suppliers_df.to_sql("suppliers", conn, if_exists="replace", index=False)

conn.close()


In [5]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.utilities.sql_database import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///piadineria.db")

from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

sql_toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000191B5DB3F80>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000191B5DB3F80>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x00000191B5DB3F80>),
 QuerySQLCheckerTool(description='Use this tool to 

## Add to cart tool

In [7]:
# Define the tool to add an item to the cart
from langchain.tools import tool

@tool
def add_to_cart(item_name: str, item_price: float) -> str:
    """Add an item to the cart."""
    url = 'http://localhost:3000/cart'  # Ensure this matches the JSON Server endpoint
    cart_item = {
        'name': item_name,
        'price': item_price
    }
    
    response = requests.post(url, json=cart_item)
    
    if response.status_code == 201:
        return f"Item '{item_name}' added to cart successfully."
    else:
        return f"Failed to add item to cart: {response.status_code} {response.text}"


# Retriever tool

In [9]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS


from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
import time
from langchain_text_splitters import CharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader



file_path = (
    "documents"
)
loader = PyPDFDirectoryLoader(file_path)
raw_documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunked_documents = text_splitter.split_documents(raw_documents)

index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [10]:
vector_store.add_documents(chunked_documents)

retriever = vector_store.as_retriever()

from langchain_core.tools.retriever import create_retriever_tool

rag_tool = create_retriever_tool(
    retriever,
    "document_search",
    """
    Search and return information restaurants health certificate and owner's history.
    """
)

## Prompt Design

In [ ]:
# Define the prompt template
prompt = ("""You are an AI assistant for a Piadineria Restaurant. Interact in english.
            You help customers explore the menu and choose the best piadine or Italian specialties through friendly, interactive questions.
            When the user asks for product details (ingredients, allergens, vegetarian options, price, etc.), you can query the product database.

            Once the user is ready to order, ask if they'd like to add the selected item to their cart.
            If they confirm, add the item to the cart using your tools.

            When using a tool, respond only with the final result. For example:
            Human: Add Classic Piadina to the cart with price 5.50
            AI: Item 'Classic Piadina' added to cart successfully.
        """
)

## Running the Agent

https://smith.langchain.com/o/77bc0d8c-7dec-4b89-8656-81cffecbaeab

In [12]:
import os
load_dotenv()
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "askmamma"



In [13]:
# With LANGSMITH_TRACING=true and LANGSMITH_API_KEY set, AgentExecutor auto-traces
# all agent invocations to LangSmith. No manual LangChainTracer needed.

# Verify tracing is configured:
print(f"Tracing enabled: {os.environ.get('LANGSMITH_TRACING')}")
print(f"Project: {os.environ.get('LANGSMITH_PROJECT')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT')}")

Tracing enabled: true
Project: askmamma
Endpoint: https://api.smith.langchain.com


In [14]:
# Setup the toolkit
toolkit = [rag_tool, add_to_cart, *sql_toolkit.get_tools()[:4]]


In [16]:
from langchain.tools import tool
from langchain.agents import create_agent

agent = create_agent(llm, toolkit, system_prompt=prompt)

In [17]:
from langchain.messages import AIMessage, HumanMessage

for chunk in agent.stream({
    "messages": [{"role": "user", "content": "do you have food certificates?"}]
}, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        if isinstance(latest_message, HumanMessage):
            print(f"User: {latest_message.content}")
        elif isinstance(latest_message, AIMessage):
            print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

User: do you have food certificates?
Calling tools: ['document_search']
Agent: Yes. We operate under a documented **HACCP** food-safety system, and we also follow an **ISO 22000-aligned Food Safety Management System**.

Additionally, all our food handlers complete **accredited local food hygiene training** (and we appoint a trained **Person In Charge (PIC)** at each location). Our HACCP plan is **reviewed/audited quarterly**, and we keep monitoring and training records **available upon request**.


In [19]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

print("🤖 Chat session started! Type 'exit' to quit.\n")

while True:
    # 1) Get user query
    user_query = input("🧑‍💬 You: ")
    if user_query.strip().lower() == "exit":
        print("👋 Goodbye!")
        break

    # 2) Prepare agent inputs
    inputs = {
        "messages": [
            {"role": "user", "content": user_query}
        ]
    }

    print("\n🤖 Agent response (streaming):")

    # 3) Stream agent output
    for chunk in agent.stream(inputs, stream_mode="values"):
        # Each chunk contains the updated agent state after each step
        latest_message = chunk["messages"][-1]

        if latest_message.content:
            # Human message (if it popped up)
            if isinstance(latest_message, HumanMessage):
                print(f"🧑‍💬 You: {latest_message.content}")

            # AI message
            elif isinstance(latest_message, AIMessage):
                print(f"🤖 Agent: {latest_message.content}")

        elif getattr(latest_message, "tool_calls", None):
            # Tool calls and arguments
            calls = [tc["name"] for tc in latest_message.tool_calls]
            print(f"🛠️ Calling tool(s): {calls}")

    print("\n" + "—" * 40 + "\n")

🤖 Chat session started! Type 'exit' to quit.


🤖 Agent response (streaming):
🧑‍💬 You: hi
🤖 Agent: Hi! Welcome to our piadineria. Are you in the mood for a classic meat piadina, something vegetarian, or maybe a lighter option (like bresaola/rucola)?

If you tell me:
1) meat / veggie / no preference  
2) any allergies (gluten, dairy, etc.)  
I’ll suggest the best options.

————————————————————————————————————————


🤖 Agent response (streaming):
🧑‍💬 You: do you have ricotta cheese in stock
🛠️ Calling tool(s): ['sql_db_list_tables']
🛠️ Calling tool(s): ['sql_db_schema']
🛠️ Calling tool(s): ['sql_db_query_checker']
🛠️ Calling tool(s): ['sql_db_query']
🤖 Agent: Yes — we have **Ricotta Cheese** in stock right now (**15** available).

————————————————————————————————————————

👋 Goodbye!


# (Optional) Agent output evaluation

In [17]:
from langsmith import Client, wrappers
from openai import OpenAI

ls_client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT"),
)

oai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
openai_client = wrappers.wrap_openai(oai_client)

In [ ]:
# Create (or re-use) an evaluation dataset
dataset = ls_client.create_dataset(
    dataset_name="QA Askmamma v2", description="A sample dataset in LangSmith."
)

examples = [
    {
        "inputs": {"question": "What is Piadina?"},
        "outputs": {"answer": "A traditional Italian flatbread, typically made with wheat flour, water, and salt."},
    },
    {
        "inputs": {"question": "What is the tradition of Piadina?"},
        "outputs": {"answer": "Piadina is a traditional Italian flatbread that originated in the Romagna region. It is typically filled with various ingredients and served warm."},
    },
]

ls_client.create_examples(dataset_id=dataset.id, examples=examples)

{'example_ids': ['98caefe4-85f3-4f7d-a523-c2c4fdeefb5b',
  '904e70bd-a3a5-4cae-b6e9-261c7e0d7076'],
 'count': 2,
 'as_of': '2026-03-07T15:23:16.52756134Z'}

In [27]:
def target(inputs: dict) -> dict:
    """Invoke the AgentExecutor and return the final response."""
    response = agent.invoke({"input": inputs["question"]})
    return {"response": response["messages"][-1].content}

In [19]:
# (target function is defined in the cell above — this cell intentionally left empty)

In [28]:
eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """LLM-as-judge evaluator using the (inputs, outputs, reference_outputs) signature."""
    user_content = f"""You are grading the following question:
{inputs['question']}
Here is the real answer:
{reference_outputs['answer']}
You are grading the following predicted answer:
{outputs['response']}
Respond with CORRECT or INCORRECT."""
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ],
    ).choices[0].message.content
    return response.strip().upper() == "CORRECT"

In [29]:
# Evaluate the agent against the QA dataset
experiment_results = ls_client.evaluate(
    target,
    data="QA Askmamma v2",
    evaluators=[correctness],
    experiment_prefix="first-eval-in-langsmith",
    max_concurrency=2,
)

View the evaluation results for experiment: 'first-eval-in-langsmith-35501fa3' at:
https://smith.langchain.com/o/77bc0d8c-7dec-4b89-8656-81cffecbaeab/datasets/276161d5-ff89-4f62-9fae-1f07f86db077/compare?selectedSessions=aca14389-44a8-44a3-8f86-cfa5272926ca




0it [00:00, ?it/s]

# (Optional) Agent tool evaluation

In [30]:
import uuid

questions = [
    (
        "Do you have ricotta cheese in stock",
        {
            "reference": "Yes we do have ricotta cheese in stock.",
            "expected_tools": ["sql_db_list_tables", "sql_db_schema", "sql_db_query_checker", "sql_db_query"],
        },
    ),
    (
        "hi",
        {
            "reference": "Hello, how can I assist you?",
            "expected_tools": [],  # Expect a direct response
        },
    ),
    (
        "Can you add ricotta cheese to the cart with price 5.50?",
        {
            "reference": "The item 'ricotta cheese' has been added to the cart.",
            "expected_tools": ["add_to_cart"],
        },
    ),
    (
        "Do you have food safety certificate?",
        {
            "reference": "Yes, we have a food safety certificate.",
            "expected_tools": ["document_search"],
        },
    ),
]

uid = uuid.uuid4()
dataset_name = f"Agent Tool Eval Example {uid}"
ds = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="An example agent evals dataset using search and calendar checks.",
)
ls_client.create_examples(
    inputs=[{"question": q[0]} for q in questions],
    outputs=[q[1] for q in questions],
    dataset_id=ds.id,
)

{'example_ids': ['96d48882-0948-4865-8a2a-0d0cb19e7fce',
  '3e3cf56b-01f3-498c-aa04-26600ae363f6',
  '255653e5-b28f-4ba5-a684-5f8077b3e396',
  'ef20f1b2-9a4e-4245-adbb-0f51e729115b'],
 'count': 4,
 'as_of': '2026-03-10T16:56:48.624066907Z'}

In [31]:
def tool_selection_correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Check whether the agent selected the expected tools.
    
    Uses intermediate_steps from AgentExecutor to extract tool names.
    """
    used_tools = outputs.get("tools_used", [])
    expected_tools = reference_outputs.get("expected_tools", [])
    score = int(used_tools == expected_tools)
    return {"key": "tool_selection_correctness", "score": score}

In [32]:
def agent_tool_target(inputs: dict) -> dict:
    """Target for tool evaluation — returns both the response and tools used."""
    response = agent.invoke({"input": inputs["question"]})
    # Extract tool names from AIMessage.tool_calls in the messages list
    tools_used = [
        tc['name']
        for msg in response['messages']
        if isinstance(msg, AIMessage) and getattr(msg, 'tool_calls', None)
        for tc in msg.tool_calls
    ]
    return {
        "response": response["messages"][-1].content,
        "tools_used": tools_used,
    }

In [ ]:
def qa_correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """LLM-based QA evaluator — checks if the response matches the reference."""
    user_content = f"""Question: {inputs['question']}
Reference answer: {reference_outputs['reference']}
Agent answer: {outputs['response']}
Is the agent's answer correct and consistent with the reference? Respond CORRECT or INCORRECT."""
    result = openai_client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[
            {"role": "system", "content": "You are an expert evaluator. Judge semantic correctness, not exact wording."},
            {"role": "user", "content": user_content},
        ],
    ).choices[0].message.content
    return result.strip().upper() == "CORRECT"

chain_results = ls_client.evaluate(
    agent_tool_target,
    data=dataset_name,
    evaluators=[tool_selection_correctness, qa_correctness],
    experiment_prefix="Agent Eval Example",
    max_concurrency=1,
)

View the evaluation results for experiment: 'Agent Eval Example-ccb07a8f' at:
https://smith.langchain.com/o/77bc0d8c-7dec-4b89-8656-81cffecbaeab/datasets/de4137fd-a540-4564-8e31-d0b00042e0ef/compare?selectedSessions=e3d646f9-f275-4560-b6b4-f3c0f5b90a78




0it [00:00, ?it/s]